In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchWindowException, WebDriverException
import csv
import re
import time
import requests


# ====================================================================== #
#                                CONFIG                                   #
# ====================================================================== #
# A normal Airbnb search results URL for ONE city (copy it from your
# browser's address bar after searching). Make sure it has check_in and
# check_out params already, or the site will use its own defaults.
#
# You can either:
#   (a) edit DEFAULT_TEMPLATE_URL / DEFAULT_CITY_INFO below and just run
#       the script, or
#   (b) leave them as-is and the script will ask you for a URL and city
#       name interactively when it starts.
DEFAULT_TEMPLATE_URL = "https://www.airbnb.com/s/Amsterdam--Netherlands/homes?adults=1&check_in=2026-07-22&check_out=2026-07-27"

DEFAULT_CITY_INFO = {
    "city": "Amsterdam",
    "day_type": "weekday",
}

DEFAULT_OUTPUT_FILE = "airbnb_single_city_data.csv"

MAX_PAGES = 20           # how many result pages to try (roughly 15-20 listings per page)
MAX_ROOMS_PER_PAGE = 20  # scrape every room found on the page (raise/lower as needed)

BASE_URL = "https://www.airbnb.com"


# ====================================================================== #
#                      REVERSE GEOCODING (district/state/country)         #
# ====================================================================== #
# Uses OpenStreetMap's free Nominatim service to turn a lat/lng pair into
# district, state, country_code, country_name automatically - no manual
# input needed. Nominatim's usage policy requires max 1 request/second and
# a descriptive User-Agent, both handled below. Results are cached so the
# same coordinates (rounded) are never looked up twice.
_geocode_cache = {}
_last_geocode_call_time = [0.0]  # mutable holder so it can be updated inside the function


def _is_latin_text(text):
    """Returns True if the text is composed of Latin-alphabet characters
    (plus common punctuation/digits/spaces), i.e. looks like English rather
    than Greek, Arabic, Cyrillic, etc. Used to reject district/state names
    that Nominatim returned in the local language despite accept-language=en
    (this happens when no English translation exists in OpenStreetMap for
    that specific place)."""
    if not text:
        return False
    for ch in text:
        if ch.isalpha() and not ("a" <= ch.lower() <= "z"):
            return False
    return True


def reverse_geocode(latitude, longitude):
    result = {"district": "", "state": "", "country_code": "", "country_name": ""}
    if not latitude or not longitude:
        return result

    # Round coordinates slightly so nearby listings share a cache entry
    # (reduces API calls without meaningfully hurting accuracy).
    cache_key = (round(float(latitude), 3), round(float(longitude), 3))
    if cache_key in _geocode_cache:
        return _geocode_cache[cache_key]

    # Respect Nominatim's 1-request-per-second limit.
    elapsed = time.time() - _last_geocode_call_time[0]
    if elapsed < 1.1:
        time.sleep(1.1 - elapsed)

    try:
        response = requests.get(
            "https://nominatim.openstreetmap.org/reverse",
            params={
                "lat": latitude,
                "lon": longitude,
                "format": "jsonv2",
                "accept-language": "en",  # force English names instead of local-language ones
            },
            headers={"User-Agent": "airbnb-scraper-personal-project/1.0"},
            timeout=10,
        )
        response.encoding = "utf-8"  # avoid mojibake (e.g. "Ø§Ù„Ù‚Ø§Ù‡Ø±Ø©") from a wrong default encoding
        _last_geocode_call_time[0] = time.time()
        if response.status_code == 200:
            addr = response.json().get("address", {})

            # Try each candidate field in order, but SKIP any value that
            # isn't in Latin/English script - some places simply don't have
            # an English name registered in OpenStreetMap, and Nominatim
            # falls back to the local-language name (e.g. Greek, Arabic)
            # even when accept-language=en was requested. We'd rather leave
            # the field blank than store text in the wrong language.
            district_candidates = [
                addr.get("suburb"), addr.get("city_district"), addr.get("neighbourhood"),
                addr.get("borough"), addr.get("town"),
            ]
            result["district"] = next(
                (c for c in district_candidates if c and _is_latin_text(c)), ""
            )

            state_candidates = [addr.get("state"), addr.get("region")]
            result["state"] = next(
                (c for c in state_candidates if c and _is_latin_text(c)), ""
            )

            result["country_code"] = (addr.get("country_code") or "").upper()

            country_name = addr.get("country") or ""
            result["country_name"] = country_name if _is_latin_text(country_name) else ""

            print(f"    >>> geocoded ({latitude}, {longitude}) -> {result}")
            if any(c and not _is_latin_text(c) for c in district_candidates):
                print(f"    >>> NOTE: some district candidates were in a non-English script "
                      f"and were skipped (no English name available in OpenStreetMap for this spot).")
        else:
            print(f"    >>> WARNING: geocoding request returned status {response.status_code}")
    except Exception as e:
        print(f"    >>> WARNING: geocoding failed: {e}")

    _geocode_cache[cache_key] = result
    return result


# ====================================================================== #
#                             HELPER FUNCTIONS                             #
# ====================================================================== #
def start_driver():
    print(">>> Installing/starting Chromedriver...")
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    print(">>> Chromedriver ready.")
    return driver


def get_room_links_from_search_page(driver, search_url):
    print(f"\n>>> Loading search page:\n{search_url}")
    driver.get(search_url)
    time.sleep(4)
    print(f">>> current_url = {driver.current_url}")
    print(f">>> page title  = {driver.title}")

    # Scroll repeatedly until the page height stops growing (i.e. no more
    # listings are being lazy-loaded), instead of a fixed small number of
    # scrolls. This captures more listings per page since Airbnb loads
    # cards progressively as you scroll down.
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0
    for _ in range(20):  # hard cap so we never scroll forever
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.7)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            stable_count += 1
            if stable_count >= 2:  # height stopped changing twice in a row -> done
                break
        else:
            stable_count = 0
        last_height = new_height

    try:
        WebDriverWait(driver, 15).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, "a[href*='/rooms/']")) > 0
        )
    except Exception as e:
        print(f">>> WARNING: timed out waiting for /rooms/ links: {e}")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    seen_ids = set()
    room_links = []
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if "/rooms/" not in href:
            continue
        match = re.search(r"/rooms/(\d+)", href)
        if not match:
            continue
        room_id = match.group(1)
        if room_id in seen_ids:
            continue
        seen_ids.add(room_id)
        room_links.append(room_id)

    print(f">>> Found {len(room_links)} unique room IDs on this page: {room_links}")
    return room_links


def scrape_one_room(driver, room_id, checkin_date, checkout_date):
    room_url = f"{BASE_URL}/rooms/{room_id}?check_in={checkin_date}&check_out={checkout_date}&adults=1"
    print(f"\n    >>> Loading room page: {room_url}")

    driver.get(room_url)

    # Wait until the URL actually reflects the room page (not a redirect or
    # a stale search page), before touching the DOM at all.
    try:
        WebDriverWait(driver, 15).until(
            lambda d: f"/rooms/{room_id}" in d.current_url
        )
    except Exception as e:
        print(f"    >>> WARNING: URL did not confirm /rooms/{room_id} in time: {e}")

    # Additionally wait until the page title stops being the generic Airbnb
    # homepage title, which is a sign the room page's real content (title,
    # price, etc.) hasn't rendered yet.
    try:
        WebDriverWait(driver, 15).until(
            lambda d: d.title and "Vacation rentals, cabins, beach houses" not in d.title
        )
    except Exception as e:
        print(f"    >>> WARNING: page title still generic after waiting: {e}")

    time.sleep(3)
    for _ in range(4):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.4)
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(1)

    print(f"    >>> room page title = {driver.title}")

    page_text = driver.find_element(By.TAG_NAME, "body").text
    soup = BeautifulSoup(driver.page_source, "html.parser")

    data = {
        "price_total": "", "room_type": "", "is_shared_room": False, "is_private_room": False,
        "max_guests": "", "is_superhost": False, "is_business_listing": False,
        "cleanliness_score": "", "guest_satisfaction_score": "", "bedrooms": "",
        "longitude": "", "latitude": "",
    }

    # ---- Room type (must be the LISTING title, not a map/search/footer h1) ----
    # Airbnb's room page can contain multiple <h1> tags (e.g. hidden ones
    # for map/search/footer UI). We filter out ones that look like generic
    # site boilerplate, and prefer the one that looks like an actual listing
    # title.
    BOILERPLATE_PHRASES = [
        "search result", "homes within map", "site footer", "site header",
        "vacation rentals, cabins, beach houses", "stay tuned",
    ]
    title_el = None
    candidate_h1s = soup.find_all("h1")
    for h1 in candidate_h1s:
        text = h1.get_text(strip=True)
        if not text:
            continue
        if any(phrase in text.lower() for phrase in BOILERPLATE_PHRASES):
            continue
        title_el = h1
        break
    if title_el is None:
        title_el = soup.find("h2")

    if title_el:
        room_type_text = title_el.get_text(strip=True)
        data["room_type"] = room_type_text
        lowered = room_type_text.lower()
        data["is_shared_room"] = "shared" in lowered
        data["is_private_room"] = "private" in lowered
        print(f"    >>> room_type = {room_type_text!r}")
    else:
        print("    >>> WARNING: no usable <h1> or <h2> found for room_type")

    # ---- Guests / bedrooms from page text ----
    guests_match = re.search(r"(\d+)\s*guest", page_text, re.IGNORECASE)
    bedrooms_match = re.search(r"(\d+)\s*bedroom", page_text, re.IGNORECASE)
    if guests_match:
        guests_val = int(guests_match.group(1))
        # Sanity check: a real listing won't sleep hundreds/thousands of
        # guests. Reject implausible matches (e.g. a stray "2026" from a
        # date elsewhere on the page) instead of keeping a wrong number.
        if 0 < guests_val <= 20:
            data["max_guests"] = str(guests_val)
    if bedrooms_match:
        bedrooms_val = int(bedrooms_match.group(1))
        if 0 < bedrooms_val <= 20:
            data["bedrooms"] = str(bedrooms_val)
    print(f"    >>> max_guests = {data['max_guests']!r}, bedrooms = {data['bedrooms']!r}")

    # ---- Superhost / business ----
    data["is_superhost"] = "Superhost" in page_text
    data["is_business_listing"] = "Business travel ready" in page_text
    print(f"    >>> is_superhost = {data['is_superhost']}")

    # ---- Price ----
    # Robust approach: search within the parsed HTML (not raw page text) so
    # we can specifically SKIP any price shown with a strikethrough/line-
    # through style (the original pre-discount price), and take the final,
    # actual price the guest would pay instead.
    try:
        price_pattern = re.compile(r'(?:ج\.?م\.?|£|\$|€)\s*([\d,]+(?:\.\d+)?)')

        def _has_strikethrough(tag):
            """Checks the tag and a few ancestor levels for a line-through
            style, which Airbnb uses to display the original (pre-discount)
            price next to the current discounted one."""
            current = tag
            for _ in range(5):
                if current is None or not hasattr(current, "get"):
                    break
                style = current.get("style") or ""
                classes = " ".join(current.get("class") or [])
                if "line-through" in style or "line-through" in classes or "strikethrough" in classes:
                    return True
                current = current.parent
            return False

        price_total = ""
        for candidate in soup.find_all(string=price_pattern):
            parent = candidate.parent
            if _has_strikethrough(parent):
                continue  # skip the original/struck-through price
            match = price_pattern.search(candidate)
            if match:
                price_total = match.group(1).replace(",", "")
                break  # first non-strikethrough match = the actual current price

        # Fallback: if every match happened to look struck-through (parsing
        # quirk) or none were found via soup, fall back to a plain text
        # search rather than leaving it silently blank.
        if not price_total:
            fallback_match = price_pattern.search(page_text)
            if fallback_match:
                price_total = fallback_match.group(1).replace(",", "")

        data["price_total"] = price_total
        if price_total:
            print(f"    >>> price_total = {price_total!r} (current/discounted price, strikethrough skipped)")
        else:
            print("    >>> WARNING: no price pattern matched in page text")
    except Exception as e:
        print(f"    >>> WARNING: price extraction failed: {e}")

    # ---- Cleanliness / guest satisfaction ----
    cleanliness_match = re.search(r"Cleanliness[^\d]{0,20}(\d+(?:\.\d+)?)", page_text, re.IGNORECASE)
    if cleanliness_match:
        cleanliness_val = float(cleanliness_match.group(1))
        # Sanity check: Airbnb's category ratings are on a 1-5 scale. A
        # match outside that range (e.g. 10, 33, 45) is almost certainly a
        # stray number picked up near the word "Cleanliness" rather than
        # the actual rating, so we discard it instead of keeping garbage.
        if 0 < cleanliness_val <= 5:
            data["cleanliness_score"] = cleanliness_match.group(1)
            print(f"    >>> cleanliness_score = {data['cleanliness_score']!r}")
        else:
            print(f"    >>> cleanliness_score left blank (implausible match: {cleanliness_val})")
    else:
        print("    >>> cleanliness_score left blank (no 'Cleanliness' rating text found on this page)")

    rating_match = re.search(r"(\d+\.\d+)\s*(?:out of 5|★)", page_text)
    if rating_match:
        try:
            data["guest_satisfaction_score"] = str(round(float(rating_match.group(1)) * 20, 1))
        except Exception:
            data["guest_satisfaction_score"] = rating_match.group(1)
    print(f"    >>> guest_satisfaction_score = {data['guest_satisfaction_score']!r}")

    # ---- Coordinates from embedded JSON ----
    lat_lng_match = re.search(r'"lat":\s*(-?\d+\.\d+).{0,80}?"lng":\s*(-?\d+\.\d+)', driver.page_source)
    if not lat_lng_match:
        lat_lng_match = re.search(r'"latitude":\s*(-?\d+\.\d+).{0,80}?"longitude":\s*(-?\d+\.\d+)', driver.page_source)
    if lat_lng_match:
        data["latitude"] = lat_lng_match.group(1)
        data["longitude"] = lat_lng_match.group(2)
        print(f"    >>> lat/lng = {data['latitude']}, {data['longitude']}")
    else:
        print("    >>> WARNING: no lat/lng found in page source")

    return data


# ====================================================================== #
#                                  MAIN                                    #
# ====================================================================== #
def main():
    print("=" * 70)
    print("Airbnb Single-City Scraper - interactive setup")
    print("=" * 70)

    url_input = input(
        f"\nEnter the Airbnb search URL for the city\n"
        f"(press Enter to use the default: {DEFAULT_TEMPLATE_URL}):\n> "
    ).strip()
    template_url = url_input if url_input else DEFAULT_TEMPLATE_URL

    city_input = input(
        f"\nEnter the city name for the 'city' column\n"
        f"(press Enter to use the default: {DEFAULT_CITY_INFO['city']}):\n> "
    ).strip()
    city_info = dict(DEFAULT_CITY_INFO)
    if city_input:
        city_info["city"] = city_input
    print(">>> district/state/country_code/country_name will be looked up automatically "
          "per room from its coordinates (via OpenStreetMap).")

    file_input = input(
        f"\nEnter the output CSV file name (without .csv)\n"
        f"(press Enter to use the default: {DEFAULT_OUTPUT_FILE}):\n> "
    ).strip()
    output_file = f"{file_input}.csv" if file_input else DEFAULT_OUTPUT_FILE

    print(f"\n>>> Using URL: {template_url}")
    print(f">>> Using city info: {city_info}")

    print(f"\n>>> Output will be saved to: {output_file}")
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "price_total", "room_type", "is_shared_room", "is_private_room", "max_guests",
            "is_superhost", "is_business_listing", "cleanliness_score",
            "guest_satisfaction_score", "bedrooms", "longitude", "latitude", "city", "day_type",
            "district", "state", "country_code", "country_name"
        ])

    # Airbnb uses different param names in different URL styles:
    # search pages sometimes use "checkin"/"checkout" (no underscore),
    # while room pages use "check_in"/"check_out". Support both so dates
    # are correctly picked up regardless of which style you pasted.
    checkin_match = re.search(r"check_?in=([\d-]+)", template_url)
    checkout_match = re.search(r"check_?out=([\d-]+)", template_url)
    checkin_date = checkin_match.group(1) if checkin_match else "2026-07-22"
    checkout_date = checkout_match.group(1) if checkout_match else "2026-07-27"
    print(f">>> Using check_in={checkin_date}, check_out={checkout_date}")

    driver = start_driver()
    total_rows_written = 0
    globally_seen_room_ids = set()   # dedup across ALL pages, not just within one page
    previous_page_room_ids = None    # used to detect "stuck" pagination

    try:
        for page_num in range(MAX_PAGES):
            offset = page_num * 20
            sep = "&" if "?" in template_url else "?"
            search_url = f"{template_url}{sep}items_offset={offset}"

            try:
                room_ids = get_room_links_from_search_page(driver, search_url)
            except (NoSuchWindowException, WebDriverException) as e:
                print(f"\n>>> STOPPED: the browser window was closed or crashed ({type(e).__name__}).")
                print(">>> Do not close the Chrome window manually while the script is running.")
                print(f">>> Data collected so far ({total_rows_written} rows) is already saved in {output_file}.")
                break

            if not room_ids:
                print(f">>> No rooms found on page {page_num + 1}, stopping.")
                break

            # Airbnb's items_offset parameter is no longer reliable and often
            # returns the exact same listings regardless of offset. If this
            # page's room IDs are identical to the previous page's, we're
            # stuck in a loop - stop instead of re-scraping duplicates.
            if previous_page_room_ids is not None and set(room_ids) == previous_page_room_ids:
                print(
                    f"\n>>> Page {page_num + 1} returned the exact same listings as the "
                    f"previous page (items_offset isn't changing results). Stopping here."
                )
                break
            previous_page_room_ids = set(room_ids)

            # Skip any room ID we've already scraped on an earlier page.
            new_room_ids = [rid for rid in room_ids if rid not in globally_seen_room_ids]
            skipped_count = len(room_ids) - len(new_room_ids)
            if skipped_count:
                print(f">>> Skipping {skipped_count} room(s) already scraped on a previous page.")
            for rid in new_room_ids:
                globally_seen_room_ids.add(rid)

            if not new_room_ids:
                print(f">>> All rooms on page {page_num + 1} were already scraped, stopping.")
                break

            rows = []
            browser_closed = False
            for i, room_id in enumerate(new_room_ids[:MAX_ROOMS_PER_PAGE]):
                print(f"\n>>> --- Room {i + 1}/{min(len(new_room_ids), MAX_ROOMS_PER_PAGE)} (page {page_num + 1}) ---")
                try:
                    room_data = scrape_one_room(driver, room_id, checkin_date, checkout_date)
                except (NoSuchWindowException, WebDriverException) as e:
                    print(f"\n>>> STOPPED: the browser window was closed or crashed ({type(e).__name__}).")
                    print(">>> Do not close the Chrome window manually while the script is running.")
                    browser_closed = True
                    break
                except Exception as e:
                    print(f">>> ERROR scraping room {room_id}: {e}")
                    continue

                # Skip rows with no usable room_type at all (e.g. sponsored
                # "Stay tuned" placeholder cards that aren't real listings).
                if not room_data["room_type"]:
                    print(f"    >>> Skipping room {room_id}: no valid room_type found (likely a placeholder card).")
                    continue

                geo = reverse_geocode(room_data["latitude"], room_data["longitude"])

                rows.append([
                    room_data["price_total"], room_data["room_type"], room_data["is_shared_room"],
                    room_data["is_private_room"], room_data["max_guests"], room_data["is_superhost"],
                    room_data["is_business_listing"],
                    room_data["cleanliness_score"], room_data["guest_satisfaction_score"],
                    room_data["bedrooms"], room_data["longitude"], room_data["latitude"],
                    city_info["city"], city_info["day_type"],
                    geo["district"], geo["state"], geo["country_code"], geo["country_name"],
                ])

            if rows:
                with open(output_file, "a", newline="", encoding="utf-8") as f:
                    writer = csv.writer(f)
                    writer.writerows(rows)
                total_rows_written += len(rows)
                print(f"\n>>> Wrote {len(rows)} rows from page {page_num + 1} to {output_file}")

            if browser_closed:
                break

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    print(f"\n>>> DONE. Total rows written: {total_rows_written}")
    print(f">>> Check the file: {output_file}")


if __name__ == "__main__":
    main()

Airbnb Single-City Scraper - interactive setup
